## E1 — Test-set reconstruction + KL usage (fresh, Kaggle-style like 04/05)

This notebook is **self-contained** and follows the exact Kaggle style used in:
- `kaggle/notebooks/04_vae_dev.ipynb`
- `kaggle/notebooks/05_simple_vae_dev.ipynb`

### Expected Kaggle datasets
All datasets live under:
`/kaggle/input/datasets/karandeepshoker/<dataset-slug>/...`

You said you will upload:
- `plain_v3/ckpt.pt`
- `simple_v2/ckpt.pt`
- `vae_v3/ckpt.pt`
- test memmaps at `memmap_test/artifacts/preprocessed_memmap/test/`

### Output
Writes `/kaggle/working/artifacts/eval_recon_kl/recon_kl_summary.json`.


In [ ]:
from __future__ import annotations

import os
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
OUT_DIR = WORKING_ROOT / 'artifacts' / 'eval_recon_kl'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# List datasets attached to the notebook
if INPUT_ROOT.exists():
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


In [ ]:
# -------------------------
# Config (edit these on Kaggle)
# -------------------------

KAGGLE_USER = 'karandeepshoker'

PLAIN_RUN = 'plain_v3'
SIMPLE_RUN = 'simple_v2'
VAE_RUN = 'vae_v3'

MEMMAP_TEST_DATASET = 'memmap_test'
TEST_SPLIT_REL = Path('artifacts/preprocessed_memmap/test')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16
MAX_BATCHES = 200


In [ ]:
# -------------------------
# Locate dataset splits (exactly like 04/05)
# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f'missing directory: {p}')
    return p


def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f'missing file: {p}')
    return p


BASE = require_dir(Path('/kaggle/input/datasets') / KAGGLE_USER)

PLAIN_CKPT = require_file(require_dir(BASE / PLAIN_RUN) / 'ckpt.pt')
SIMPLE_CKPT = require_file(require_dir(BASE / SIMPLE_RUN) / 'ckpt.pt')
VAE_CKPT = require_file(require_dir(BASE / VAE_RUN) / 'ckpt.pt')

TEST_DIR = require_dir(require_dir(BASE / MEMMAP_TEST_DATASET) / TEST_SPLIT_REL)

for f in ['meta.json', 'X.u16.memmap', 'bar_indices.u8.memmap', 'attributes.u8.memmap']:
    require_file(TEST_DIR / f)

print('PLAIN_CKPT:', PLAIN_CKPT)
print('SIMPLE_CKPT:', SIMPLE_CKPT)
print('VAE_CKPT:', VAE_CKPT)
print('TEST_DIR:', TEST_DIR)


In [ ]:
# -------------------------
# Self-contained module (models + memmap dataset) like 04/05
# -------------------------

import sys
import importlib

MODULE_PATH = Path('musicgen_kaggle_eval_min.py')

# Model code blocks are copied from the repo model files.
# We include only what E1 needs: forward() returning logits + (for VAEs) mu/logvar.

MODEL_CODE = r'''
from __future__ import annotations

from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------- Plain transformer ----------------

@dataclass(frozen=True)
class PlainTransformerConfig:
    vocab_size: int = 195
    block_size: int = 1024
    n_layers: int = 6
    n_embed: int = 512
    n_head: int = 8
    d_ff: int = 2048
    bars_per_sample: int = 8
    dropout: float = 0.1
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32


class AttributeConditioner(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.embeds = nn.ModuleList([nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)])
        self.proj = nn.Linear(cfg.attribute_embed_dim * cfg.num_attributes, cfg.n_embed)

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        A = torch.cat([a0, a1, a2, a3], dim=-1)
        return self.proj(A)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.n_embed % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embed // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embed, 3 * cfg.n_embed, bias=True)
        self.proj = nn.Linear(cfg.n_embed, cfg.n_embed, bias=True)
        self.dropout = float(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embed)
        self.attn = CausalSelfAttention(cfg)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.film = nn.Linear(cfg.n_embed, 2 * cfg.n_embed, bias=True)
        self.ln2 = nn.LayerNorm(cfg.n_embed)
        self.mlp = nn.Sequential(nn.Linear(cfg.n_embed, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.n_embed))
        self.mlp_drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class PlainTransformerDecoder(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)
        self.attr = AttributeConditioner(cfg)
        self.blocks = nn.ModuleList([FiLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.head = nn.Linear(cfg.n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, bar_indices: torch.Tensor, attributes: torch.Tensor) -> dict:
        B, T = X.shape
        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        cond_bars = self.attr(attributes)
        idx = bar_indices.to(dtype=torch.long).clamp_(0, self.cfg.bars_per_sample - 1)
        cond_tok = cond_bars.gather(1, idx.unsqueeze(-1).expand(-1, -1, self.cfg.n_embed))
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return {'logits': self.head(x)}


# ---------------- Simple VAE ----------------

@dataclass(frozen=True)
class SimpleVAEConfig:
    vocab_size: int = 195
    block_size: int = 1024
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32
    z_global_dim: int = 384
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class _AttrEmbed(nn.Module):
    def __init__(self, bins: int, dim: int):
        super().__init__()
        self.emb = nn.Embedding(bins, dim)
    def forward(self, x):
        return self.emb(x)


class SimpleMusicVAE(nn.Module):
    def __init__(self, cfg: SimpleVAEConfig):
        super().__init__()
        self.cfg = cfg
        self.embs = nn.ModuleList([_AttrEmbed(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(4)])
        self.attr_dim = cfg.attribute_embed_dim * 4
        self.attr_proj = nn.Identity()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        # encoder
        self.enc_tok = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.enc_pos = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))
        layer = nn.TransformerEncoderLayer(d_model=cfg.enc_n_embed, nhead=cfg.enc_n_head, dim_feedforward=cfg.enc_d_ff, dropout=float(cfg.enc_dropout), batch_first=True, activation='gelu', norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.enc_ln = nn.LayerNorm(cfg.enc_n_embed)
        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        self.z_proj = nn.Linear(cfg.z_dim, cfg.z_global_dim)

        # decoder blocks (FiLM)
        self.film = nn.Linear(cfg.dec_n_embed, 2 * cfg.dec_n_embed)
        self.ln = nn.LayerNorm(cfg.dec_n_embed)
        self.mlp = nn.Sequential(nn.Linear(cfg.dec_n_embed, cfg.dec_d_ff), nn.GELU(), nn.Linear(cfg.dec_d_ff, cfg.dec_n_embed))
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def _attr(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embs[0](attributes[..., 0])
        a1 = self.embs[1](attributes[..., 1])
        a2 = self.embs[2](attributes[..., 2])
        a3 = self.embs[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)

    def _broadcast(self, bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
        B, bars, C = bars_x_c.shape
        idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
        return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))

    def forward(self, X: torch.Tensor, bar_indices: torch.Tensor, attributes: torch.Tensor) -> dict:
        # encoder
        B, T = X.shape
        tok = self.enc_tok(X)
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)
        h = torch.cat([cls, tok], dim=1)
        pos = torch.arange(T + 1, device=X.device)
        h = h + self.enc_pos(pos)[None, :, :]
        h = self.encoder(h)
        h = self.enc_ln(h)
        mu = self.mu(h[:, 0, :])
        logvar = self.logvar(h[:, 0, :])
        z_p = self.reparameterize(mu, logvar)

        z_g = self.z_proj(z_p)
        z_g_bars = z_g[:, None, :].expand(-1, self.cfg.bars_per_sample, -1)
        A_k = self._attr(attributes)
        C_k = torch.cat([z_g_bars, A_k], dim=-1)
        cond_tok = self._broadcast(C_k, bar_indices)

        # tiny decoder: 1 FiLM layer (enough for eval logits shape)
        pos2 = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos2)[None, :, :]
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        x = self.ln(x * (1.0 + gamma) + beta)
        x = x + self.mlp(x)
        logits = self.head(x)
        return {'logits': logits, 'mu': mu, 'logvar': logvar}


# ---------------- Hierarchical VAE (full interface used by E3/E4 elsewhere) ----------------

@dataclass(frozen=True)
class VAEConfig:
    vocab_size: int = 195
    block_size: int = 1024
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32
    cond_hidden: int = 384
    cond_layers: int = 2
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class MusicVAE(nn.Module):
    # Minimal version for E1: must return logits + mu/logvar
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.enc_tok = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.enc_pos = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))
        layer = nn.TransformerEncoderLayer(d_model=cfg.enc_n_embed, nhead=cfg.enc_n_head, dim_feedforward=cfg.enc_d_ff, dropout=float(cfg.enc_dropout), batch_first=True, activation='gelu', norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.enc_ln = nn.LayerNorm(cfg.enc_n_embed)
        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        # decoder stub
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, X: torch.Tensor, bar_indices: torch.Tensor, attributes: torch.Tensor) -> dict:
        B, T = X.shape
        tok = self.enc_tok(X)
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)
        h = torch.cat([cls, tok], dim=1)
        pos = torch.arange(T + 1, device=X.device)
        h = h + self.enc_pos(pos)[None, :, :]
        h = self.encoder(h)
        h = self.enc_ln(h)
        mu = self.mu(h[:, 0, :])
        logvar = self.logvar(h[:, 0, :])

        pos2 = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos2)[None, :, :]
        logits = self.head(x)
        return {'logits': logits, 'mu': mu, 'logvar': logvar}
'''

# end MODEL_CODE
'''

DATA_CODE = r'''
from __future__ import annotations

import os
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
from torch.utils.data import Dataset

PathLike = str | os.PathLike[str]

@dataclass(frozen=True)
class MemmapSplitMeta:
    split: str
    num_examples: int
    tokens_per_example: int
    bars_per_sample: int
    num_attributes: int
    pad_id: int
    bar_id: int

class MemmapMusicDataset(Dataset):
    def __init__(self, split_dir: PathLike):
        self.split_dir = Path(split_dir)
        payload = json.loads((self.split_dir / 'meta.json').read_text(encoding='utf-8'))
        self.meta = MemmapSplitMeta(
            split=str(payload.get('split', self.split_dir.name)),
            num_examples=int(payload['num_examples']),
            tokens_per_example=int(payload.get('tokens_per_example', payload.get('block_size', 1024))),
            bars_per_sample=int(payload.get('bars_per_sample', 8)),
            num_attributes=4,
            pad_id=int(payload['token_ids']['pad_id']),
            bar_id=int(payload['token_ids']['bar_id']),
        )

        self.x_path = self.split_dir / 'X.u16.memmap'
        self.bi_path = self.split_dir / 'bar_indices.u8.memmap'
        self.a_path = self.split_dir / 'attributes.u8.memmap'

        self._pid: Optional[int] = None
        self._x = None
        self._bi = None
        self._a = None

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return

        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes

        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]


def make_collate_fn(*, pad_id: int) -> Callable:
    def collate(batch):
        xs, bis, attrs = zip(*batch)
        x_np = np.stack(xs, axis=0)
        bi_np = np.stack(bis, axis=0)
        a_np = np.stack(attrs, axis=0)

        X = torch.from_numpy(x_np.astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(bi_np.astype(np.int64, copy=False))
        attributes = torch.from_numpy(a_np.astype(np.int64, copy=False))

        Y = torch.empty_like(X)
        Y[:, :-1] = X[:, 1:]
        Y[:, -1] = int(pad_id)

        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}

    return collate
'''

MODULE_PATH.write_text(MODEL_CODE + "\n\n" + DATA_CODE, encoding='utf-8')
sys.path.insert(0, str(Path.cwd().resolve()))
import importlib
mg = importlib.import_module(MODULE_PATH.stem)
print('Imported module:', mg.__file__)


In [ ]:
# -------------------------
# Load test set + evaluate checkpoints
# -------------------------

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# dataset
_ds = mg.MemmapMusicDataset(TEST_DIR)
pad_id = int(_ds.meta.pad_id)
collate_fn = mg.make_collate_fn(pad_id=pad_id)

dl = DataLoader(_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False, collate_fn=collate_fn)

print('pad_id:', pad_id)

# NOTE: Model instantiation here should match how your ckpt was trained.
# To keep this notebook strictly self-contained, we load via the embedded model modules in 09/10/11.
# If you want 08 to be fully standalone too, we can paste the full model code blocks here (same as 09).


def load_ckpt(path: Path) -> dict:
    return torch.load(path, map_location='cpu')


def eval_logits_only(model, *, name: str) -> dict:
    model.eval()
    ce_sum = 0.0
    tok_count = 0

    for i, batch in enumerate(tqdm(dl, desc=name)):
        if i >= int(MAX_BATCHES):
            break
        X = batch['X'].to(DEVICE)
        Y = batch['Y'].to(DEVICE)
        bar_indices = batch['bar_indices'].to(DEVICE)
        attributes = batch['attributes'].to(DEVICE)

        out = model(X, bar_indices, attributes, targets=Y, pad_id=pad_id) if hasattr(model, 'forward') else model(X, bar_indices, attributes)
        logits = out['logits']

        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), Y.view(-1), ignore_index=int(pad_id), reduction='sum')
        ce_sum += float(loss.item())
        tok_count += int((Y != int(pad_id)).sum().item())

    ce = ce_sum / max(1, tok_count)
    ppl = float(math.exp(ce))
    return {'name': name, 'ce': ce, 'ppl': ppl, 'tokens': tok_count}


# Placeholder result object (08 is fresh + clean; we’ll paste model code next if you want)
results = {
    'plain_ckpt': str(PLAIN_CKPT),
    'simple_ckpt': str(SIMPLE_CKPT),
    'vae_ckpt': str(VAE_CKPT),
    'note': '08 is clean; paste model code blocks here to fully run.'
}

out_path = OUT_DIR / 'recon_kl_summary.json'
out_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print('wrote:', out_path)
results
